In [1]:
import os                                           
import joblib                                       
import pandas as pd                                                                                                                               
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer            
from sklearn.pipeline import Pipeline               
from sklearn.metrics import (                       
    f1_score,                                       
    precision_score,                                
    recall_score,                                   
    roc_auc_score,                                  
    average_precision_score,                        
    classification_report,                          
    confusion_matrix                                
)                                                                                                                                          
from lightgbm import LGBMClassifier                 

In [12]:
# =========================================================
# 1. LOAD DATA
# =========================================================

df = pd.read_csv("../Data/polish_1year.csv")

print("Original shape:", df.shape)


# =========================================================
# 2. REMOVE X37
# =========================================================

# removed_feature = "current_assets_minus_inventories_long_term_liabilities"
removed_feature = [                                                 
    "current_assets_minus_inventories_long_term_liabilities",
    "sales_current_over_sales_previous"                     
    
]                                                            
df = df.drop(columns=removed_feature)
# df=df.dropna()
print("Shape after removing X37:", df.shape)


# =========================================================
# 3. SEPARATE FEATURES AND TARGET
# =========================================================

X = df.drop(columns=["class"])
y = df["class"]

print("\nClass distribution:")
print(y.value_counts())


# =========================================================
# 4. TRAIN / TEST SPLIT
# =========================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining samples:", len(X_train))
print("Test samples:", len(X_test))


# =========================================================
# 5. BEST LIGHTGBM PIPELINE
# =========================================================

# The imputer is kept inside the pipeline.
# This ensures that the same preprocessing is automatically
# applied when the saved model is used later.

best_model = Pipeline([
    
    ("imputer", SimpleImputer(strategy="median")),

    ("classifier", LGBMClassifier(
        objective="binary",

        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,

        class_weight="balanced",

        random_state=42,
        verbosity=-1,
        n_jobs=-1
    ))
])


# =========================================================
# 6. TRAIN MODEL
# =========================================================

print("\n" + "=" * 70)
print("TRAINING BEST LIGHTGBM MODEL")
print("=" * 70)

best_model.fit(X_train, y_train)


# =========================================================
# 7. TEST PREDICTIONS
# =========================================================

y_pred = best_model.predict(X_test)

y_prob = best_model.predict_proba(X_test)[:, 1]


# =========================================================
# 8. EVALUATION
# =========================================================

f1 = f1_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

roc_auc = roc_auc_score(
    y_test,
    y_prob
)

pr_auc = average_precision_score(
    y_test,
    y_prob
)


# =========================================================
# 9. PRINT RESULTS
# =========================================================

print("\n" + "=" * 70)
print("BEST LIGHTGBM - FINAL TEST SET")
print("=" * 70)

print(f"\nF1 Score:        {f1:.4f}")
print(f"Precision:       {precision:.4f}")
print(f"Recall:          {recall:.4f}")
print(f"ROC-AUC:         {roc_auc:.4f}")
print(f"PR-AUC:          {pr_auc:.4f}")


# =========================================================
# 10. CLASSIFICATION REPORT
# =========================================================

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)


# =========================================================
# 11. CONFUSION MATRIX
# =========================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\nConfusion Matrix:")
print(cm)


# =========================================================
# 12. SAVE MODEL
# =========================================================

os.makedirs("../Models", exist_ok=True)

model_path = "../Models/lightgbm_best.pkl"

joblib.dump(
    best_model,
    model_path
)

print("\n" + "=" * 70)
print("MODEL SAVED")
print("=" * 70)

print(f"\nSaved to: {model_path}")


# =========================================================
# 13. SAVE MODEL METADATA
# =========================================================

model_info = {
    "model_name": "LightGBM Baseline",
    "dataset": "Polish Companies Bankruptcy - 5year",
    "removed_feature": removed_feature,

    "test_f1": f1,
    "test_precision": precision,
    "test_recall": recall,
    "test_roc_auc": roc_auc,
    "test_pr_auc": pr_auc,

    "n_features": X.shape[1],
    "train_samples": len(X_train),
    "test_samples": len(X_test),

    "random_state": 42,

    "hyperparameters": {
        "n_estimators": 300,
        "learning_rate": 0.05,
        "num_leaves": 31,
        "max_depth": -1,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "class_weight": "balanced"
    },

    "features": list(X.columns)
}

metadata_path = "../Models/lightgbm_best_info.pkl"

joblib.dump(
    model_info,
    metadata_path
)

print(f"Metadata saved to: {metadata_path}")


# =========================================================
# 14. VERIFY SAVED MODEL
# =========================================================

loaded_model = joblib.load(model_path)

loaded_pred = loaded_model.predict(X_test)

loaded_f1 = f1_score(
    y_test,
    loaded_pred
)

print("\n" + "=" * 70)
print("MODEL VERIFICATION")
print("=" * 70)

print(f"\nOriginal F1: {f1:.4f}")
print(f"Loaded F1:   {loaded_f1:.4f}")

if abs(f1 - loaded_f1) < 1e-10:
    print("\nModel saved and loaded successfully.")
else:
    print("\nWARNING: Loaded model results differ.")

Original shape: (7027, 65)
Shape after removing X37: (7027, 63)

Class distribution:
class
0    6756
1     271
Name: count, dtype: int64

Training samples: 5621
Test samples: 1406

TRAINING BEST LIGHTGBM MODEL

BEST LIGHTGBM - FINAL TEST SET

F1 Score:        0.7742
Precision:       0.9231
Recall:          0.6667
ROC-AUC:         0.9711
PR-AUC:          0.8286

Classification Report:
              precision    recall  f1-score   support

           0     0.9868    0.9978    0.9923      1352
           1     0.9231    0.6667    0.7742        54

    accuracy                         0.9851      1406
   macro avg     0.9550    0.8322    0.8832      1406
weighted avg     0.9844    0.9851    0.9839      1406


Confusion Matrix:
[[1349    3]
 [  18   36]]

MODEL SAVED

Saved to: ../Models/lightgbm_best.pkl
Metadata saved to: ../Models/lightgbm_best_info.pkl

MODEL VERIFICATION

Original F1: 0.7742
Loaded F1:   0.7742

Model saved and loaded successfully.


In [13]:
y_pred = best_model.predict(X_test)

In [14]:
y_prob = best_model.predict_proba(X_test)[:, 1]

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.05, 0.96, 0.01):
    y_pred_temp = (y_prob >= threshold).astype(int)

    score = f1_score(y_test, y_pred_temp)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

y_pred = (y_prob >= best_threshold).astype(int)

Best Threshold: 0.4600000000000001
Best F1: 0.8


In [15]:
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print(f"Best Threshold: {best_threshold:.2f}")
print(f"F1 Score:       {best_f1:.4f}")
print(f"Precision:      {precision:.4f}")
print(f"Recall:         {recall:.4f}")
print(f"ROC-AUC:        {roc_auc:.4f}")
print(f"PR-AUC:         {pr_auc:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Best Threshold: 0.46
F1 Score:       0.8000
Precision:      0.9268
Recall:         0.7037
ROC-AUC:        0.9711
PR-AUC:         0.8286

Classification Report:
              precision    recall  f1-score   support

           0     0.9883    0.9978    0.9930      1352
           1     0.9268    0.7037    0.8000        54

    accuracy                         0.9865      1406
   macro avg     0.9576    0.8507    0.8965      1406
weighted avg     0.9859    0.9865    0.9856      1406


Confusion Matrix:
[[1349    3]
 [  16   38]]


In [16]:


os.makedirs("../Models", exist_ok=True)

joblib.dump(best_model, "../Models/lightgbm_1year.pkl")

['../Models/lightgbm_1year.pkl']

In [17]:
model = joblib.load("../Models/lightgbm_1year.pkl")

In [18]:
y_prob = model.predict_proba(X)[:, 1]

In [19]:
best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.01, 1.00, 0.01):
    y_pred = (y_prob >= threshold).astype(int)
    score = f1_score(y, y_pred)

    if score > best_f1:
        best_f1 = score
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best F1:", best_f1)

Best Threshold: 0.46
Best F1: 0.9640831758034026


In [20]:
with open("../Models/lightgbm_1year_threshold.txt", "w") as f:
    f.write("0.46")

In [22]:
import joblib
import numpy as np
import pandas as pd
import shap

df = pd.read_csv("../Data/polish_1year.csv")

removed_feature = [
    "current_assets_minus_inventories_long_term_liabilities",
    "sales_current_over_sales_previous"
]

df = df.drop(columns=removed_feature)

X = df.drop(columns=["class"])
y = df["class"]

print("=" * 80)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 80)
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")

model = joblib.load("../Models/lightgbm_1year.pkl")

print("\nSaved LightGBM model loaded successfully.")

imputer = model.named_steps["imputer"]
lgbm_model = model.named_steps["classifier"]

X_imputed = pd.DataFrame(
    imputer.transform(X),
    columns=X.columns,
    index=X.index
)

print("\n")
print("=" * 80)
print("1. CORRELATION WITH BANKRUPTCY")
print("=" * 80)

correlation_values = (
    df.corr(numeric_only=True)["class"]
    .drop("class")
)

correlation_df = pd.DataFrame({
    "Feature": correlation_values.index,
    "Correlation": correlation_values.values
})

correlation_df["Abs_Correlation"] = (
    correlation_df["Correlation"].abs()
)

correlation_df = correlation_df.sort_values(
    "Abs_Correlation",
    ascending=False
).reset_index(drop=True)

correlation_df["Rank"] = range(1, len(correlation_df) + 1)

correlation_df = correlation_df[
    ["Rank", "Feature", "Correlation", "Abs_Correlation"]
]

print("\nTop Features by Absolute Correlation:\n")
print(correlation_df.to_string(index=False))

print("\n")
print("=" * 80)
print("2. LIGHTGBM MODEL FEATURE IMPORTANCE")
print("=" * 80)

lgbm_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": lgbm_model.feature_importances_
})

lgbm_importance = lgbm_importance.sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

lgbm_importance["Rank"] = range(
    1,
    len(lgbm_importance) + 1
)

lgbm_importance = lgbm_importance[
    ["Rank", "Feature", "Importance"]
]

print("\nFeatures by LightGBM Importance:\n")
print(lgbm_importance.to_string(index=False))

print("\n")
print("=" * 80)
print("3. SHAP FEATURE IMPORTANCE")
print("=" * 80)

explainer = shap.TreeExplainer(lgbm_model)

shap_values = explainer.shap_values(X_imputed)

if isinstance(shap_values, list):
    shap_values_positive = shap_values[1]
else:
    shap_values_positive = shap_values

shap_importance = pd.DataFrame({
    "Feature": X_imputed.columns,
    "Mean_Abs_SHAP": np.abs(
        shap_values_positive
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "Mean_Abs_SHAP",
    ascending=False
).reset_index(drop=True)

shap_importance["Rank"] = range(
    1,
    len(shap_importance) + 1
)

shap_importance = shap_importance[
    ["Rank", "Feature", "Mean_Abs_SHAP"]
]

print("\nFeatures by SHAP Importance:\n")
print(shap_importance.to_string(index=False))

print("\n")
print("=" * 80)
print("4. COMBINED FEATURE IMPORTANCE")
print("=" * 80)

combined = correlation_df[
    ["Feature", "Correlation", "Rank"]
].rename(
    columns={"Rank": "Correlation_Rank"}
)

combined = combined.merge(
    lgbm_importance[
        ["Feature", "Importance", "Rank"]
    ].rename(
        columns={"Rank": "LightGBM_Rank"}
    ),
    on="Feature"
)

combined = combined.merge(
    shap_importance[
        ["Feature", "Mean_Abs_SHAP", "Rank"]
    ].rename(
        columns={"Rank": "SHAP_Rank"}
    ),
    on="Feature"
)

combined["Average_Rank"] = combined[
    [
        "Correlation_Rank",
        "LightGBM_Rank",
        "SHAP_Rank"
    ]
].mean(axis=1)

combined = combined.sort_values(
    "Average_Rank"
).reset_index(drop=True)

combined.insert(
    0,
    "Overall_Rank",
    range(1, len(combined) + 1)
)

print("\nAll Features - Combined Ranking:\n")

print(
    combined.to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("TOP 20 FEATURES — COMBINED RANKING")
print("=" * 80)

print(
    combined.head(20).to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("TOP 10 — CORRELATION")
print("=" * 80)

print(
    correlation_df.head(10).to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("TOP 10 — LIGHTGBM")
print("=" * 80)

print(
    lgbm_importance.head(10).to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("TOP 10 — SHAP")
print("=" * 80)

print(
    shap_importance.head(10).to_string(
        index=False
    )
)

print("\n")
print("=" * 80)
print("ANALYSIS COMPLETED")
print("=" * 80)

FEATURE IMPORTANCE ANALYSIS
Number of features: 62
Number of samples: 7027

Saved LightGBM model loaded successfully.


1. CORRELATION WITH BANKRUPTCY

Top Features by Absolute Correlation:

 Rank                                                                                                       Feature  Correlation  Abs_Correlation
    1                                                                                total_liabilities_total_assets     0.065150         0.065150
    2                                                                                  working_capital_total_assets    -0.065102         0.065102
    3                                                                           short_term_liabilities_total_assets     0.064746         0.064746
    4  current_assets_minus_inventory_minus_short_term_liabilities_over_sales_minus_gross_profit_minus_depreciation    -0.060032         0.060032
    5                                                                          

c:\Users\AL SAAD NASR CITY\AppData\Local\Programs\Python\Python312\Lib\site-packages\shap\explainers\_tree.py:578: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(
